

# Load employment data and calculate ratio
# ...

In [1]:
import os
import pandas as pd
import requests
from urllib3.exceptions import InsecureRequestWarning
from urllib3 import disable_warnings
from difflib import SequenceMatcher

# Disable SSL warnings (not recommended for production)
disable_warnings(InsecureRequestWarning)

# Define the URL and download path
url_fuente = 'https://www.economia.gob.ar/download/infoeco/apendice3a.xlsx'
download_path = './../data/info/apendice3a.xlsx'

# Download the Excel file
response = requests.get(url_fuente, verify=False)
with open(download_path, "wb") as file:
    file.write(response.content)

# Load the sheets needed
EPH_puntual = pd.read_excel(download_path, sheet_name='EPH Puntual')
TD = pd.read_excel(download_path, sheet_name='TD 03-')

In [2]:
import numpy as np

# Select the sheet to work with
datos = TD  # or use EPH_puntual for other data
serie = '45.2_'

# Preprocess the data
datos = datos.iloc[:, :]  # Adjust to include only relevant columns

# Mask to find rows containing the series identifier
mask = datos.apply(lambda x: x.astype(str).str.contains(serie))

# Filter rows and clean the data
df = datos[mask.cumsum() == True].dropna(axis=1, how='all').dropna()
df, df.columns = df[1:], df.iloc[0]  # Adjust column names

# Process the time column
col_tiempo = df.columns[0]
df[col_tiempo] = df[col_tiempo].str.split('.').apply(lambda x: x[1] + x[0])
df[col_tiempo] = df[col_tiempo].str.replace(' ', '')
df[col_tiempo] = df[col_tiempo].str.replace('III', 'Q3').str.replace('II', 'Q2').str.replace('IV', 'Q4').str.replace('I', 'Q1')
df[col_tiempo] = pd.to_datetime(df[col_tiempo])

# Set the time column as index
df = df.set_index(col_tiempo)
df = df.replace('s/d', np.nan).astype(float).round(4)

# Find the shared root name across all columns
raiz_ = df.columns[0]
for col in df.columns:
    match = SequenceMatcher(None, raiz_, col).find_longest_match(0, len(raiz_), 0, len(col))
    next_ = raiz_[match.a:match.a + match.size]
    raiz_ = next_


/tmp/ipykernel_60517/2193173699.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col_tiempo] = pd.to_datetime(df[col_tiempo])
/tmp/ipykernel_60517/2193173699.py:26: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('s/d', np.nan).astype(float).round(4)


In [3]:


# Function to shift dates to the middle of the second month of the quarter
def shift_to_mid_quarter(d):
    if d.month == 1:
        return pd.Timestamp(year=d.year, month=2, day=15)
    elif d.month == 4:
        return pd.Timestamp(year=d.year, month=5, day=15)
    elif d.month == 7:
        return pd.Timestamp(year=d.year, month=8, day=15)
    elif d.month == 10:
        return pd.Timestamp(year=d.year, month=11, day=15)

# Set the new index to df
df.index = df.index.map(shift_to_mid_quarter)


# Functions that can be used... 

# from dateutil.relativedelta import relativedelta
# from datetime import datetime

import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
from funciones import shift_to_mid_quarter_index, resample_and_interpolate


# Ensure the index is datetime type
df.index = pd.to_datetime(df.index)

# Shift the index to the 15th of the second month of the quarter
df = shift_to_mid_quarter_index(df)

def resample_and_interpolate(df, freq='Q-FEB'):
    """
    Resample the DataFrame to the specified frequency and interpolate missing values.
    """
    df_resampled = df.resample(freq).ffill().bfill()
    df_resampled.index = df_resampled.index.map(lambda x: pd.Timestamp(year=x.year, month=x.month, day=15))
    df_resampled = df_resampled.interpolate(method='linear')
    df_resampled.fillna(df.mean(), inplace=True)
    return df_resampled

# # # Resample and interpolate the data
df_resampled = resample_and_interpolate(df)
# df_resampled = df.interpolate(method='linear')


# Save the processed data to CSV
output_file = f'./../data/info/{raiz_}.csv'
df_resampled.to_csv(output_file)

print(f"Data saved to {output_file}")

# Now df's index should match empleo's index


Data saved to ./../data/info/45.2_ECTDT.csv


/tmp/ipykernel_60517/1786805108.py:37: FutureWarning: 'Q-FEB' is deprecated and will be removed in a future version, please use 'QE-FEB' instead.
  df_resampled = df.resample(freq).ffill().bfill()
